# 06 — Exportação final

**Objetivo:** montar os cinco GeoPackages finais (um por cidade, geometria exata de 2022), arredondando as
contagens interpoladas sem alterar os totais municipais, e acompanhá-los do dicionário de colunas.

**Entradas:**
- `output/05_derivadas/<slug_cidade>_completo.gpkg` (5 arquivos)
- `output/01_diagnostico/VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx`

**Saídas:**
- `output/final/<Cidade>_setores_compatibilizados_2022.gpkg` (5 arquivos)
- `output/final/dicionario_saida.csv`
- `output/final/VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx`

**Depende de:** `05_variaveis_derivadas.ipynb`

## 1. Setup

In [ ]:
import shutil
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd

from utils import CIDADES, ANOS, OUTPUT_DIR, VARS_EXTENSIVAS, VAR_RENDA

IN_DIR = OUTPUT_DIR / "05_derivadas"
OUT_DIR = OUTPUT_DIR / "final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 2. Arredondamento com preservação de totais

- As contagens de `_2000` e `_2010` saem fracionárias da ponderação de área; a entrega usa inteiros.
- Método do **maior resto**: arredonda para baixo e distribui a sobra entre as feições de maior parte fracionária.
- Premissa: preserva exatamente o total municipal, ao contrário de `round()` feição a feição.

In [ ]:
def arredonda_preservando_total(serie):
    '''Arredonda uma série para inteiros, preservando a soma total (método do maior resto).'''
    if serie.isna().all():
        return serie
    valores = serie.fillna(0).to_numpy()
    total_alvo = int(round(valores.sum()))
    pisos = np.floor(valores).astype(int)
    resto_a_distribuir = total_alvo - pisos.sum()
    fracoes = valores - pisos

    resultado = pisos.copy()
    if resto_a_distribuir > 0:
        indices_maior_fracao = np.argsort(-fracoes)[:resto_a_distribuir]
        resultado[indices_maior_fracao] += 1
    elif resto_a_distribuir < 0:
        indices_menor_fracao = np.argsort(fracoes)[:-resto_a_distribuir]
        resultado[indices_menor_fracao] -= 1

    resultado = resultado.astype(float)
    resultado[serie.isna().to_numpy()] = np.nan
    return pd.Series(resultado, index=serie.index)


import numpy as np


## 3. Montagem dos GeoPackages finais

In [ ]:
for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    gdf = gpd.read_file(IN_DIR / f"{slug}_completo.gpkg")

    for ano in (2000, 2010):
        for var in VARS_EXTENSIVAS:
            col = f"{var}_{ano}"
            if col in gdf.columns:
                gdf[col] = arredonda_preservando_total(gdf[col])

    nome_arquivo = f"{cidade}_setores_compatibilizados_2022.gpkg"
    gdf.to_file(OUT_DIR / nome_arquivo, driver="GPKG")
    print(f"{cidade}: {len(gdf)} setores, {gdf.shape[1]} colunas -> {nome_arquivo}")


In [ ]:
resumo_final = []
for cidade in CIDADES:
    gdf = gpd.read_file(OUT_DIR / f"{cidade}_setores_compatibilizados_2022.gpkg")
    resumo_final.append({
        "cidade": cidade, "n_setores_2022": len(gdf),
        "pop_2000": int(gdf["V1_2000"].sum()), "pop_2010": int(gdf["V1_2010"].sum()),
        "pop_2022": int(gdf["V1_2022"].sum()),
    })
pd.DataFrame(resumo_final)


## 4. Dicionário de colunas de saída

Explica a origem de cada coluna (contagem original, interpolada, derivada ou renda deflacionada), com as
ressalvas conceituais que precisam acompanhar a entrega.

In [ ]:
linhas_dicionario = [
    {"coluna": "CD_SETOR", "origem": "original 2022", "descricao": "Código do setor censitário — geometria e chave da malha de 2022, inalterada."},
    {"coluna": "area_m2", "origem": "calculada", "descricao": "Área do setor (malha 2022), em m², recalculada no CRS métrico da cidade."},
]

for var in VARS_EXTENSIVAS + [VAR_RENDA, "V24A"] + [f"V{n}" for n in (2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 23, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55)]:
    for ano in ANOS:
        col = f"{var}_{ano}"
        if var == "V24A" and ano != 2022:
            continue
        if var == VAR_RENDA:
            origem = "original (nominal)" if ano == 2022 else "interpolada (média ponderada por V26), nominal"
        elif ano == 2022:
            origem = "original 2022"
        elif var in VARS_EXTENSIVAS:
            origem = f"interpolada de {ano} para a malha 2022 (ponderação de área, arredondada preservando total)"
        else:
            origem = f"derivada (fórmula da planilha corrigida), calculada sobre as contagens de {ano}"
        linhas_dicionario.append({"coluna": col, "origem": origem, "descricao": ""})

for ano in (2000, 2010):
    linhas_dicionario.append({
        "coluna": f"{VAR_RENDA}_{ano}_r2022", "origem": "deflacionada (IPCA, base jul/2022)",
        "descricao": f"{VAR_RENDA}_{ano} convertida para reais de julho/2022.",
    })

df_dicionario_saida = pd.DataFrame(linhas_dicionario)
df_dicionario_saida.to_csv(OUT_DIR / "dicionario_saida.csv", index=False, encoding="utf-8-sig")
df_dicionario_saida.head(15)


## 5. Cópia da planilha de fórmulas corrigida

In [ ]:
origem_xlsx = OUTPUT_DIR / "01_diagnostico" / "VARIAVEIS_censos_2000_2010_2022_CORRIGIDA.xlsx"
shutil.copy(origem_xlsx, OUT_DIR / origem_xlsx.name)
print("Planilha corrigida copiada para", OUT_DIR / origem_xlsx.name)
